In [ ]:
# Put this in ONE notebook cell in notebooks/fraud_baseline.ipynb and run it.
# It is self-contained: if X_train etc. exist already it uses them; otherwise it will load the CSV and make splits.

import os
import numpy as np
import pandas as pd
import joblib
import random
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, classification_report, confusion_matrix
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ---------- 1) Load data & make train/test if not already present ----------
if 'X_train' not in globals():
    print("Loading data and creating train/test split...")
    df = pd.read_csv("../data/creditcard.csv")
    X = df.drop(columns=["Class"])
    y = df["Class"].astype(int)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=SEED
    )
else:
    print("Using existing X_train/X_test from notebook kernel")

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

# ---------- 2) Define preprocessing (ColumnTransformer) ----------
num_cols = X_train.columns.tolist()   # Time, V1..V28, Amount
preproc = ColumnTransformer([("scale", StandardScaler(), num_cols)], remainder='drop')

# ---------- 3) Create pipelines ----------
pipe_logreg = Pipeline([
    ("preproc", preproc),
    ("clf", LogisticRegression(solver="saga", class_weight="balanced", max_iter=10000, random_state=SEED))
])

pipe_rf = Pipeline([
    ("preproc", preproc),
    ("clf", RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=SEED, n_jobs=-1))
])

# SMOTE pipeline (training-only: we will NOT save SMOTE in inference)
pipe_smote_rf = ImbPipeline([
    ("preproc", preproc),
    ("smote", SMOTE(random_state=SEED)), # <-- REMOVED n_jobs
    ("clf", RandomForestClassifier(n_estimators=100, random_state=SEED))
])

pipelines = {
    "logreg": pipe_logreg,
    "rf": pipe_rf,
    "smote_rf": pipe_smote_rf
}

# ---------- 4) Fit pipelines (this may take a minute) ----------
for name, p in pipelines.items():
    print(f"Fitting {name} ...")
    p.fit(X_train, y_train)
    print(f"Done fitting {name}")

# ---------- 5) Evaluate and pick best (by Average Precision - PR AUC) ----------
def get_proba(pipe, X):
    # RF and LR have predict_proba, others might have decision_function
    if hasattr(pipe, "predict_proba"):
        return pipe.predict_proba(X)[:,1]
    elif hasattr(pipe, "decision_function"):
        # decision_function gives scores we can use for AUC/AP
        return pipe.decision_function(X)
    else:
        raise ValueError("Model has neither predict_proba nor decision_function")

scores = {}
for name, p in pipelines.items():
    y_proba = get_proba(p, X_test)
    ap = average_precision_score(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    scores[name] = {"ap": ap, "auc": auc}
    print(f"{name} -> AP: {ap:.4f}, ROC AUC: {auc:.4f}")

# Select best by AP
best_name = max(scores.keys(), key=lambda k: scores[k]["ap"])
final_model = pipelines[best_name]
print(f"Selected best model: {best_name} (AP={scores[best_name]['ap']:.4f})")

# ---------- 6) Build inference pipeline (NO SMOTE) ----------
# We must extract the fitted preprocessor and classifier from final_model and create a new sklearn Pipeline
if hasattr(final_model, "named_steps"):
    # For both sklearn.Pipeline and imblearn.Pipeline, named_steps includes keys 'preproc', 'smote' (maybe), 'clf'
    if 'preproc' in final_model.named_steps and 'clf' in final_model.named_steps:
        fitted_preproc = final_model.named_steps['preproc']
        fitted_clf = final_model.named_steps['clf']
    else:
        # fallback: take first step as preproc and last step as clf
        fitted_preproc = final_model.steps[0][1] if hasattr(final_model, 'steps') else final_model.named_steps[list(final_model.named_steps.keys())[0]]
        fitted_clf = final_model.steps[-1][1] if hasattr(final_model, 'steps') else final_model.named_steps[list(final_model.named_steps.keys())[-1]]
else:
    # If not a pipeline (unlikely), just assume final_model itself is classifier and we need to create preproc from scratch (not ideal)
    raise RuntimeError("Final model is not a pipeline; please ensure final_model is a Pipeline with 'preproc' and 'clf' steps.")

from sklearn.pipeline import Pipeline as SKPipeline
inference_pipe = SKPipeline([
    ("preproc", fitted_preproc),
    ("clf", fitted_clf)
])

# ---------- 7) Sanity test inference_pipe on X_test ----------
y_proba_inf = get_proba(inference_pipe, X_test)
print("Inference pipeline AP:", average_precision_score(y_test, y_proba_inf))
print("Inference pipeline ROC AUC:", roc_auc_score(y_test, y_proba_inf))

# ---------- 8) Save inference pipeline ----------
os.makedirs("../models/sklearn", exist_ok=True)
save_path = "../models/sklearn/fraud_pipeline.joblib"
joblib.dump(inference_pipe, save_path)
print("Saved inference pipeline to:", save_path)

# ---------- 9) Optional: show classification report using threshold 0.5 ----------
y_pred = (y_proba_inf >= 0.5).astype(int)
print("\nClassification report for inference_pipe (threshold 0.5):\n")
print(classification_report(y_test, y_pred, digits=4))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))


Loading data and creating train/test split...
Train shape: (227845, 30) Test shape: (56962, 30)
Fitting logreg ...


In [ ]:
import joblib, os 
pipe_path = "../models/sklearn/fraud_pipeline.joblib"
print("Exists:", os.path.exists(pipe_path))
pipe = joblib.load(pipe_path)

# Quick test on a few rows
sample = X_test.iloc[:5]
probs = pipe.predict_proba(sample)[:,1]
preds = (probs >= 0.5).astype(int)
print("probs:", probs)
print("preds:", preds)


In [3]:
from sklearn.pipeline import Pipeline as SKPipeline
import os, joblib

# Build inference pipeline from the best model
fitted_preproc = final_model.named_steps['preproc']
fitted_clf = final_model.named_steps['clf']

inference_pipe = SKPipeline([
    ("preproc", fitted_preproc),
    ("clf", fitted_clf)
])

# Save the pipeline
os.makedirs("../models/sklearn", exist_ok=True)
joblib.dump(inference_pipe, "../models/sklearn/fraud_pipeline.joblib")
print("✅ Model saved to ../models/sklearn/fraud_pipeline.joblib")


NameError: name 'final_model' is not defined